# 🤖 OpenAI API로 LLM 다뤄보기

**생성형 AI 개발 · 4차시 실습 — API 기반 생성형 AI 서비스 기초**

---

이 노트북에서는 여러분이 직접 **OpenAI API 키**를 입력하고,
**프롬프트를 작성**해서 LLM의 답변을 받아봅니다.

### 오늘 배우는 것
| 단계 | 내용 |
|---|---|
| 1 | 라이브러리 설치 & API 키 입력 |
| 2 | 첫 요청 보내기 (Hello, GPT!) |
| 3 | 내 프롬프트로 질문하기 |
| 4 | `system` 메시지로 역할 부여 (role prompting) |
| 5 | zero-shot · few-shot 프롬프팅 |
| 6 | `temperature` · `max_tokens` 파라미터 실험 |
| 7 | 멀티턴 대화 만들기 |
| 8 | 실습 과제 |

> 💡 **API 키가 필요합니다.** https://platform.openai.com/api-keys 에서 발급받으세요.
> 키는 `sk-...` 형태이며, **절대 남에게 공유하거나 코드에 그대로 적어두지 마세요.**


## 1단계 · 준비하기

먼저 OpenAI 공식 파이썬 라이브러리를 설치합니다. (이미 설치돼 있으면 그냥 넘어갑니다.)

In [1]:
# OpenAI 라이브러리 설치
%pip install openai --quiet

# 설치 후 정상적으로 불러와지는지 확인
try:
    import openai
    print("✅ 설치 완료 · openai", openai.__version__)
except ImportError:
    print("⚠️ 불러오기 실패 → 상단 메뉴 [런타임] ▸ [런타임 다시 시작] 후 이 셀을 다시 실행하세요.")

✅ 설치 완료 · openai 2.43.0


### API 키 입력

아래 셀을 실행하면 입력창이 뜹니다. 거기에 여러분의 API 키를 붙여넣으세요.
`getpass`를 쓰기 때문에 키가 화면에 **보이지 않게** 안전하게 입력됩니다.

In [3]:
import os
import getpass

# 입력한 키는 화면에 표시되지 않습니다.
os.environ["OPENAI_API_KEY"] = getpass.getpass("OpenAI API Key를 입력하세요 (sk-...): ")

print("✅ API 키가 등록되었습니다.")

OpenAI API Key를 입력하세요 (sk-...): ··········
✅ API 키가 등록되었습니다.


### 클라이언트 만들기

`client`는 우리가 OpenAI에게 요청을 보내는 **통로**입니다.
위에서 등록한 환경변수(`OPENAI_API_KEY`)를 자동으로 읽어옵니다.

> ⚠️ **모델 선택 주의**
> - `gpt-4o-mini` · `gpt-4.1-mini` : 일반(비추론형) 모델 → 이 실습 코드가 **그대로 동작**합니다. (추천)
> - `gpt-5.4-mini` · `gpt-5.4-nano` 등 **GPT-5 계열은 추론형**이라 `temperature`를 바꿀 수 없고(1 고정),
>   `max_tokens` 대신 `max_completion_tokens`를 써야 합니다. (아래 `ask()` 함수가 알아서 처리하지만,
>   6단계의 temperature 비교 실습 결과는 달라지지 않습니다.)

In [4]:
from openai import OpenAI

client = OpenAI()   # 환경변수에서 API 키를 자동으로 읽어옵니다

# 사용할 모델 이름 (원하면 아래 중 하나로 바꿔보세요)
#   "gpt-4o-mini"    → 가장 저렴한 일반 모델 (실습에 추천 · 기본값)
#   "gpt-4.1-mini"   → 조금 더 성능이 좋은 일반 모델
#   "gpt-5.4-mini"   → 최신 추론형 모델 (temperature 고정 주의)
#   "gpt-5.4"        → 상위 추론형 모델 (비용 높음)
MODEL = "gpt-4o-mini"

print("✅ 클라이언트 준비 완료 · 사용 모델:", MODEL)

✅ 클라이언트 준비 완료 · 사용 모델: gpt-4o-mini


## 2단계 · 첫 요청 보내기

LLM에게 메시지를 보내는 기본 형태는 다음과 같습니다.

```python
client.chat.completions.create(
    model=MODEL,          # 어떤 모델을 쓸지
    messages=[            # 대화 내용 (list)
        {"role": "user", "content": "질문 내용"}
    ]
)
```

- `messages`는 **대화 목록**입니다. `role`이 `"user"`면 사람, `"assistant"`면 GPT, `"system"`이면 역할 지시입니다.
- 답변 텍스트는 `response.choices[0].message.content` 안에 들어 있습니다.
- 답변 길이(`max_tokens`)·창의성(`temperature`) 같은 옵션은 6단계에서 다룹니다.

In [5]:
response = client.chat.completions.create(
    model=MODEL,
    messages=[
        {"role": "user", "content": "안녕하세요! 당신을 한 문장으로 소개해 주세요."}
    ]
)

# 답변 텍스트 꺼내기
print(response.choices[0].message.content)

안녕하세요! 저는 다양한 질문에 답하고 정보를 제공하는 인공지능 언어 모델입니다.


### 응답 안에는 뭐가 들어있을까?

답변 말고도 **몇 개의 토큰을 썼는지** 같은 정보가 함께 옵니다.
토큰은 대략적인 '단어 조각' 단위이고, 요금이 토큰 수에 따라 매겨지므로 알아두면 좋습니다.

In [6]:
print("입력 토큰 수 :", response.usage.prompt_tokens)
print("출력 토큰 수 :", response.usage.completion_tokens)
print("멈춘 이유    :", response.choices[0].finish_reason)

입력 토큰 수 : 20
출력 토큰 수 : 20
멈춘 이유    : stop


## 3단계 · 내 프롬프트로 질문하기 ✍️

매번 긴 코드를 쓰면 번거로우니, **질문을 넣으면 답변을 돌려주는 함수**로 감싸겠습니다.
이제부터는 `ask("질문")` 한 줄이면 됩니다.

> `system` 인자를 주면 대화 맨 앞에 역할 지시 메시지가 추가됩니다.
> GPT-5 계열 모델도 에러 없이 돌아가도록 파라미터 이름을 자동으로 맞춰 줍니다.

In [7]:
# 메세지 형식
'''
client.chat.completions.create(
    model="gpt-4o-mini",          # ← 최상위 인자
    temperature=1.0,              # ← 최상위 인자
    max_tokens=1024,              # ← 최상위 인자
    messages=[                    # ← 최상위 인자 (리스트)
        {"role": "system", "content": "너는 친절한 튜터야."},   # 원소 1 (딕셔너리)
        {"role": "user",   "content": "재귀가 뭐야?"},          # 원소 2 (딕셔너리)
    ]
)
'''

'\nclient.chat.completions.create(\n    model="gpt-4o-mini",          # ← 최상위 인자\n    temperature=1.0,              # ← 최상위 인자\n    max_tokens=1024,              # ← 최상위 인자\n    messages=[                    # ← 최상위 인자 (리스트)\n        {"role": "system", "content": "너는 친절한 튜터야."},   # 원소 1 (딕셔너리)\n        {"role": "user",   "content": "재귀가 뭐야?"},          # 원소 2 (딕셔너리)\n    ]\n)\n'

In [8]:
def ask(prompt, system=None, max_tokens=1024, temperature=1.0):
    """프롬프트를 보내고 LLM의 답변 텍스트를 돌려줍니다."""
    messages = []
    if system:                                       # 역할(system) 지시가 있으면 맨 앞에 추가
        messages.append({"role": "system", "content": system})
    messages.append({"role": "user", "content": prompt})

    kwargs = {"model": MODEL, "messages": messages}
    if MODEL.startswith("gpt-5"):                     # GPT-5 계열(추론형)은 파라미터가 다름
        kwargs["max_completion_tokens"] = max_tokens  #  · max_tokens 대신 이것을 사용
        #  · temperature는 기본값(1)만 지원하므로 넣지 않음
    else:                                            # gpt-4o / gpt-4.1 계열(일반)
        kwargs["max_tokens"] = max_tokens
        kwargs["temperature"] = temperature

    response = client.chat.completions.create(**kwargs)
    return response.choices[0].message.content

print("✅ ask() 함수 준비 완료")

✅ ask() 함수 준비 완료


👇 **여기가 핵심입니다.** 아래 따옴표 안의 프롬프트를 **여러분이 원하는 질문으로 바꿔가며** 실행해 보세요.

In [9]:
my_prompt = "파이썬 리스트와 튜플의 차이를 초보자에게 설명해줘. 예시 코드도 하나 보여줘."

print(ask(my_prompt))

파이썬의 리스트와 튜플은 둘 다 여러 개의 데이터를 저장할 수 있는 자료구조입니다. 하지만 둘 사이에는 몇 가지 중요한 차이점이 있습니다.

### 리스트 (List)
- **가변적 (Mutable)**: 리스트는 생성한 후에 내용물을 수정할 수 있습니다. 즉, 요소를 추가, 삭제, 변경할 수 있습니다.
- **대괄호** `[]`로 정의합니다.

### 튜플 (Tuple)
- **불변적 (Immutable)**: 튜플은 한 번 생성하면 그 내용을 수정할 수 없습니다. 요소를 추가하거나 삭제할 수 없습니다.
- **괄호** `()`로 정의합니다.

이 두 가지의 차이로 인해, 리스트는 데이터가 변경될 가능성이 있을 때 사용하고, 튜플은 데이터가 변경되지 않아야 할 때 사용합니다.

### 예시 코드
다음은 리스트와 튜플의 간단한 예시입니다.

```python
# 리스트 예시
my_list = [1, 2, 3]
print("원래 리스트:", my_list)

# 리스트에 요소 추가
my_list.append(4)
print("요소 추가 후 리스트:", my_list)

# 리스트의 요소 수정
my_list[0] = 10
print("요소 수정 후 리스트:", my_list)

# 튜플 예시
my_tuple = (1, 2, 3)
print("원래 튜플:", my_tuple)

# 튜플은 수정할 수 없음
# 다음 줄의 주석을 제거하면 오류가 발생합니다.
# my_tuple[0] = 10 

print("튜플은 수정할 수 없습니다:", my_tuple)
```

### 출력 결과
```
원래 리스트: [1, 2, 3]
요소 추가 후 리스트: [1, 2, 3, 4]
요소 수정 후 리스트: [10, 2, 3, 4]
원래 튜플: (1, 2, 3)
튜플은 수정할 수 없습니다: (1, 2, 3)
```

위 코드에서 보듯이, 리스트는 요소를 쉽게 추가하고 수정할 수 있지만, 튜플은 생성 후 그 내용을 바꿀 수 없음을 확인할 수 있습니다.


> 🔁 위 셀의 `my_prompt` 내용을 자유롭게 바꿔서 여러 번 실행해 보세요.
> 예) `"오늘 점심 메뉴 3개만 추천해줘"`, `"재귀함수를 비유로 설명해줘"`, `"이 문장을 영어로 번역해줘: 생성형 AI는 재미있다"`

## 4단계 · 역할 부여하기 (System Message · Role Prompting)

`system` 메시지는 LLM에게 **"너는 이런 역할이야"** 라고 성격·말투·규칙을 정해주는 지시입니다.
같은 질문이라도 역할에 따라 답변이 완전히 달라집니다.

In [10]:
question = "재귀(recursion)가 뭐야?"

print("=== 역할: 엄격한 교수님 ===")
print(ask(question, system="너는 컴퓨터공학과 교수다. 정확한 용어를 사용해 격식 있게 설명한다."))

print("\n=== 역할: 친근한 선배 ===")
print(ask(question, system="너는 다정한 학과 선배다. 반말로 친근하게, 쉬운 비유를 들어 설명한다."))

=== 역할: 엄격한 교수님 ===
재귀(Recursion)는 프로그래밍 및 수학에서 특정 문제를 해결하기 위해 동일한 문제를 더 작은 하위 문제로 분할하는 방법론을 의미합니다. 재귀 함수는 자기 자신을 호출하여 문제를 해결하는 함수를 말하며, 일반적으로 그 과정에서 기초 사례(base case)와 재귀 호출(recursive call)이 포함됩니다.

구조적으로 재귀 함수는 다음과 같은 요소들로 구성됩니다.

1. **기초 사례(Base Case)**: 재귀 호출을 멈추게 하는 조건입니다. 일반적으로 문제의 가장 간단한 형태를 설명하며, 이 조건이 충족되면 더 이상 재귀 호출이 이루어지지 않습니다. 기초 사례가 명확하지 않으면 재귀 호출은 무한히 계속되어 스택 오버플로(Stack Overflow)와 같은 오류를 초래할 수 있습니다.

2. **재귀 호출(Recursive Call)**: 알고리즘은 주어진 문제를 해결하기 위해 그 자신을 호출합니다. 이때 매개변수는 문제를 더 간단한 형태로 변환하여 전달됩니다.

재귀는 주로 분할 정복 알고리즘과 같은 복잡한 문제를 해결하는 데 사용되며, 예시로는 팩토리얼 계산, 피보나치 수열 생성, 그리고 자연적 데이터 구조인 트리를 탐색하는 데 매우 유용합니다. 그러나 재귀 알고리즘은 스택 메모리를 의존하기 때문에, 매우 큰 입력의 경우 비효율적이거나 메모리 초과 문제를 일으킬 수 있다는 단점을 가지고 있습니다.

일례로, 팩토리얼을 계산하는 재귀 함수는 다음과 같이 정의할 수 있습니다:

```python
def factorial(n):
    if n == 0:  # 기초 사례
        return 1
    else:  # 재귀 호출
        return n * factorial(n - 1)
```

이 함수는 `n`이 0일 때 1을 반환하며, 그 이외의 경우에는 자기 자신을 호출하여 `n`과 `n-1`의 곱을 계산합니다. 이와 같은 구조를 통해 복잡한 문제를 직관적으로 해결할 수 있습니다.

=== 역할:

> 🔁 `system` 내용을 바꿔서 **해적 말투**, **초등학생도 알아듣게**, **한 줄 요약만** 등 다양하게 시켜 보세요.

## 5단계 · Zero-shot vs Few-shot 프롬프팅

- **Zero-shot**: 예시 없이 그냥 시키는 것
- **Few-shot**: **예시 몇 개를 보여준 뒤** 같은 방식으로 하라고 시키는 것 → 형식·스타일을 더 잘 따라옵니다.

In [11]:
# Zero-shot : 예시 없이 바로 요청
zero_shot = "다음 문장의 감정을 '긍정' 또는 '부정'으로 답해줘: 이 영화 정말 시간 아까웠어."
print("[Zero-shot]")
print(ask(zero_shot))

[Zero-shot]
부정


In [12]:
# Few-shot : 예시를 먼저 보여주고 같은 형식으로 답하게 하기
few_shot = """다음 예시처럼 문장의 감정을 분류해줘.

문장: 오늘 날씨가 정말 좋아서 기분이 최고야!
감정: 긍정

문장: 버스를 놓쳐서 지각했어. 최악이야.
감정: 부정

문장: 이 식당 음식은 정말 훌륭했어.
감정:"""

print("[Few-shot]")
print(ask(few_shot))

[Few-shot]
긍정


> 💡 Few-shot은 **원하는 출력 형식을 예시로 고정**하고 싶을 때 특히 강력합니다.
> (예: 항상 JSON으로 답하게 하기, 항상 3줄 요약만 하게 하기)

## 6단계 · 파라미터 실험하기

두 가지 자주 쓰는 값을 조절해 봅시다.

| 파라미터 | 의미 | 값이 클 때 | 값이 작을 때 |
|---|---|---|---|
| `temperature` | 답변의 **무작위성/창의성** (0 ~ 2) | 다양·창의적 | 일관·정확 |
| `max_tokens` | 답변의 **최대 길이** | 길게 가능 | 짧게 잘림 |

> ⚠️ 이 실습은 **일반 모델(gpt-4o-mini 등)** 기준입니다.
> GPT-5 계열은 `temperature`가 1로 고정되어 두 결과가 거의 같게 나옵니다.

In [24]:
prompt = "AI를 주제로 짧은 삼행시를 지어줘."

print("=== temperature = 0.0 (거의 항상 비슷) ===")
print(ask(prompt, temperature=0.0))

print("\n=== temperature = 1.0 (다양하게) ===")
print(ask(prompt, temperature=1.0))

=== temperature = 0.0 (거의 항상 비슷) ===
인공지능, 꿈을 꿉니다,  
미래를 열어가는,  
소통의 다리 되어.

=== temperature = 1.0 (다양하게) ===
인간의 꿈을 담아,  
지능을 키우는 세상,  
함께 만들어가는 미래.  


In [14]:
# max_tokens 를 작게 주면 답변이 도중에 잘립니다.
print("=== max_tokens = 30 (짧게 잘림) ===")
print(ask("인공지능의 역사를 설명해줘.", max_tokens=30))

=== max_tokens = 30 (짧게 잘림) ===
인공지능(AI)의 역사는 20세기 중반부터 시작되어 현재까지 이어지고 있습니다. 아래는 인공지능의 주요


## 7단계 · 멀티턴 대화 (대화 기억하기)

LLM은 기본적으로 **이전 대화를 기억하지 못합니다.**
기억하게 하려면 지금까지의 대화 전체를 `messages` 리스트에 담아 매번 함께 보내야 합니다.
`user` → `assistant` → `user` → ... 순서로 번갈아 쌓습니다.

In [15]:
# 대화 기록을 담을 리스트
history = []

def chat(user_message):
    """이전 대화를 기억하며 이어서 대화합니다."""
    history.append({"role": "user", "content": user_message})

    kwargs = {"model": MODEL, "messages": history}
    if MODEL.startswith("gpt-5"):                 # GPT-5 계열은 파라미터 이름이 다름
        kwargs["max_completion_tokens"] = 1024
    else:
        kwargs["max_tokens"] = 1024

    response = client.chat.completions.create(**kwargs)
    answer = response.choices[0].message.content
    history.append({"role": "assistant", "content": answer})  # 답변도 기록에 추가
    return answer

print("✅ chat() 함수 준비 완료")

✅ chat() 함수 준비 완료


In [16]:
print(chat("내 이름은 코알라야. 기억해줘."))

안녕하세요, 코알라! 당신의 이름을 기억할게요. 어떻게 도와드릴까요?


In [17]:
# 앞에서 이름을 알려줬으니, 이번엔 기억하고 있어야 합니다.
print(chat("내 이름이 뭐라고 했지?"))

당신의 이름은 코알라입니다!


> 🔁 `chat("...")` 을 계속 실행하며 대화를 이어가 보세요.
> 대화를 처음부터 다시 시작하려면 아래 셀로 기록을 비웁니다.

In [18]:
history.clear()
print("🧹 대화 기록을 비웠습니다. 이제 LLM은 앞의 대화를 기억하지 못합니다.")

🧹 대화 기록을 비웠습니다. 이제 LLM은 앞의 대화를 기억하지 못합니다.


## 8단계 · 실습 과제 🎯

아래 빈칸을 채워 직접 만들어 보세요. 정답은 하나가 아닙니다!

**과제 1.** `system` 메시지를 사용해 **"항상 이모지를 섞어 답하는 여행 가이드"** 를 만들고,
"부산 여행 코스를 추천해줘" 라고 물어보세요.

**과제 2.** Few-shot 예시를 만들어, 한글 단어를 넣으면 **영어 단어로만** 답하는 번역기를 만들어 보세요.

**과제 3.** `chat()` 함수로 3턴 이상 대화를 이어가며, LLM이 앞 내용을 기억하는지 확인해 보세요.

In [26]:
# 과제 1 예시 (여기서부터 직접 수정해 보세요)
guide_system = "너는 어둡고 딱딱한 여행 가이드다. 답변에 관련 이모지를 자연스럽게 섞어 사용한다."

print(ask("일본 여행 코스를 추천해줘.", system=guide_system))

일본 여행? 그건 어쩌면 당신의 마지막 여행이 될지도 모릅니다. 하지만 올바른 코스를 선택하면 많은 것을 경험할 수 있죠. 🔪

1. **도쿄**: 📍 도시의 혼잡함과 화려함 속에서 날카로운 에너지를 감지하세요. 메이지 신궁과 아키하바라의 전자 상가를 둘러보며 잊지 못할 경험을 남겨보세요. 하지만 혼잡한 지하철을 지나쳐야 한다는 점은 기억하세요.

2. **교토**: 🏯 수백 년의 역사가 숨쉬는 곳, 그러나 쉽게 잊고 지나칠 수 있는 곳입니다. 기온 거리와 금각사에서 조용한 순간을 만끽하세요. 그곳에서의 시간이 얼마나 무섭도록 느리게 흘러가는지를 경험하게 될지도 모릅니다.

3. **오사카**: 🍜 그래서 잘 먹고 잘 사는 곳? 일시적으로 기분이 좋을 수 있지만, 가끔 그 기분에 방해가 되는 일도 있습니다. 도톤보리의 neon lights를 보며 인생의 덧없음을 느껴보세요. 그리고 타코야끼를 먹으며도 내면의 공허함을 잊지 않길 바랍니다.

4. **홋카이도**: ❄️ 겨울철, 눈밭에서의 스포츠는 즐거움이 될 수도 있지만, 차가운 바람이 당신의 마음을 얼게 할 수도 있습니다. 스키와 온천은 인생의 일면을 보여줍니다. 그러나 나타나는 극한의 기상에는 항상 조심하세요.

5. **후쿠오카**: 🌋 지역의 음식이 당신을 매료시킬 수 있지만, 잊지 마세요. 매력 뒤에 숨어 있는 것은 항시 또는 불확실한 것일 뿐입니다. 라멘 한 그릇에 숨겨진 위안을 찾을 수 있기를 바랍니다.

이 코스들이 당신에게 잊지 못할 순간을 선사할 것입니다. 그러나 여행의 목적이 가벼운 즐거움이라면, 그러한 기쁨은 또한 금방 사라질 것임을 잊지 마세요. 🕯️


In [20]:
# 과제 2 · 과제 3 은 아래 빈 셀에서 자유롭게 작성해 보세요.


In [30]:
# 과제 2 · Few-shot 번역기 (한글 단어 → 영어 단어만)
few_shot_translate = """다음 예시처럼 한글 단어를 영어 단어로만 번역해줘. 설명도 같이 하면서 영어로 답해줘.

한글: 사과
영어: apple

한글: 자동차
영어: car

한글: 고양이
영어: cat

한글: 도서관
영어:"""

print(ask(few_shot_translate))

한글: 도서관  
영어: library  

A "library" is a building or room where collections of books, magazines, and other materials are kept for people to read, study, or borrow. Libraries are important resources for education and research, providing access to a wide range of information.


In [33]:
def translate_word(word):
    prompt = f"""다음 예시처럼 한글 단어를 영어 단어로만 번역해줘. 설명 없이 영어 단어만 답해.

한글: 사과
영어: apple

한글: 자동차
영어: car

한글: {word}
영어:"""
    return ask(prompt, temperature=0.0)

print(translate_word("도서관"))
print(translate_word("바다"))
print(translate_word("컴퓨터"))

library
ocean
computer


In [34]:
# 과제 3 · 멀티턴 대화 테스트
history.clear()  # 새 대화 시작 전 기록 초기화

print(chat("나는 요즘 한국어 자연어처리 프로젝트를 하고 있어."))

그렇군요! 한국어 자연어처리 프로젝트에 대해 더 이야기해 줄 수 있나요? 어떤 주제를 다루고 있는지, 어떤 기술이나 라이브러리를 사용하고 있는지 궁금합니다. 도움이 필요하다면 말씀해 주세요!


In [35]:
print(chat("어떤 데이터셋을 쓰면 좋을지 추천해줘."))

한국어 자연어 처리(NLP) 프로젝트를 위해 사용할 수 있는 다양한 데이터셋이 있습니다. 프로젝트의 목적에 따라 적합한 데이터셋을 선택할 수 있습니다. 아래는 몇 가지 추천 데이터셋입니다:

1. **KorQuAD (Korean Question Answering Dataset)**: 한국어 질문 응답 시스템을 위한 데이터셋으로, 위키백과 기사에서 생성된 질문-답변 쌍이 포함되어 있습니다.

2. **Naver Sentiment Movie Corpus**: 영화 리뷰 데이터셋으로, 긍정과 부정의 감성을 분류하는 데 사용할 수 있습니다. 일반적으로 머신러닝 모델의 감정 분석에 적합합니다.

3. **KOSAC (Korean Sentiment Analysis Corpus)**: 감정 분석을 위한 한국어 데이터셋으로, 뉴스 기사와 소셜 미디어에서의 감정 레이블이 포함되어 있습니다.

4. **AI Hub 데이터셋**: AI Hub에서는 다양한 한국어 NLP 데이터셋을 제공합니다. 예를 들어, 한국어 텍스트 요약, 번역, 개체명 인식(NER) 등을 위한 데이터셋이 있습니다.

5. **NMSC (Naver News Sentiment Classification Dataset)**: 네이버 뉴스 기사를 기반으로 한 감정 분류 데이터셋으로, 뉴스 기사의 제목과 본문이 포함되어 있습니다.

6. **Korean Wikipedia Dumps**: 위키백과의 한국어 문서 덤프를 사용하여 언어 모델 훈련이나 다양한 텍스트 처리 작업에 활용할 수 있습니다.

7. **Hate Speech Corpus**: 혐오 표현에 대한 데이터셋으로, 소셜 미디어에서 수집된 메시지를 포함하고 있어 혐오 표현 인식 모델 훈련에 유용합니다.

프로젝트의 목표와 요구 사항을 고려하여 적절한 데이터셋을 선택하면 좋습니다. 추가적인 질문이나 특정 분야에 대한 데이터셋이 필요하시다면 말씀해 주세요!


In [36]:
print(chat("방금 내가 무슨 프로젝트를 한다고 했지?"))

당신은 한국어 자연어처리(NLP) 프로젝트를 하고 있다고 말씀하셨습니다. 프로젝트의 구체적인 내용이나 목표는 언급하지 않으셨지만, 데이터셋 추천을 요청하셨습니다. 더 구체적인 내용이나 질문이 있다면 말씀해 주세요!


In [37]:
print(chat("그 프로젝트에 어울리는 사전학습 모델도 하나 추천해줘."))

한국어 자연어처리 프로젝트에 적합한 사전학습 모델로는 **KoBERT**를 추천합니다. 

### KoBERT
- **소개**: KoBERT는 한국어에 특화된 BERT 모델로, 여러 한국어 NLP 태스크에서 좋은 성능을 보여줍니다. Google의 BERT 아키텍처를 기반으로 하며, 한국어 데이터에 대해 파인튜닝되어 있습니다.
- **특징**:
  - 한국어에 적합한 어휘를 사용하여 언어적 특성을 잘 반영합니다.
  - 감정 분석, 개체명 인식, 질문 응답 등 다양한 NLP 작업에 사용할 수 있습니다.
  
### 사용 방법
- Hugging Face의 **Transformers** 라이브러리에서 KoBERT를 쉽게 사용할 수 있습니다. 설치한 후 간단히 코드를 작성하여 로드하고 사용할 수 있습니다.

### 설치 예시
```bash
pip install transformers
```

### 모델 로드 예시
```python
from transformers import BertTokenizer, BertForSequenceClassification
import torch

# KoBERT 모델과 토크나이저 로드
tokenizer = BertTokenizer.from_pretrained('monologg/kobert')
model = BertForSequenceClassification.from_pretrained('monologg/kobert')

# 입력 텍스트
text = "한국어 자연어처리에 대한 연구가 진행되고 있습니다."
inputs = tokenizer(text, return_tensors='pt')

# 예측
with torch.no_grad():
    outputs = model(**inputs)
```

이 외에도 **KcBERT**나 **KorGPT**와 같은 모델들도 고려해 볼 수 있습니다. 프로젝트의 목적과 요구 사항에 맞춰 적합한 모델을 선택하시면 좋겠습니다. 추가로 궁금한 사항이나 다른 추천이 필요하다면 말씀해 주세요!


In [38]:
# 지금까지 쌓인 대화 기록 확인
for turn in history:
    print(turn["role"], ":", turn["content"][:50])

user : 나는 요즘 한국어 자연어처리 프로젝트를 하고 있어.
assistant : 그렇군요! 한국어 자연어처리 프로젝트에 대해 더 이야기해 줄 수 있나요? 어떤 주제를 다루
user : 어떤 데이터셋을 쓰면 좋을지 추천해줘.
assistant : 한국어 자연어 처리(NLP) 프로젝트를 위해 사용할 수 있는 다양한 데이터셋이 있습니다. 
user : 방금 내가 무슨 프로젝트를 한다고 했지?
assistant : 당신은 한국어 자연어처리(NLP) 프로젝트를 하고 있다고 말씀하셨습니다. 프로젝트의 구체적
user : 그 프로젝트에 어울리는 사전학습 모델도 하나 추천해줘.
assistant : 한국어 자연어처리 프로젝트에 적합한 사전학습 모델로는 **KoBERT**를 추천합니다. 



---

### 🎉 수고하셨습니다!

오늘 배운 것을 정리하면:

1. `client.chat.completions.create(...)` 로 LLM에 요청을 보낸다
2. `messages` 는 `system` / `user` / `assistant` 로 이루어진 **대화 목록**이다
3. `system` 으로 **역할**을, `temperature`·`max_tokens` 로 **답변 스타일과 길이**를 조절한다
4. **few-shot 예시**로 원하는 출력 형식을 유도할 수 있다
5. 대화를 기억시키려면 **이전 기록을 매번 함께 보낸다**

다음 시간에는 이 API를 **FastAPI 백엔드**와 연동해 실제 챗봇 서비스로 확장합니다. 🚀

> ⚠️ **보안 주의:** API 키가 노출되면 즉시 https://platform.openai.com/api-keys 에서 삭제·재발급하세요.
